# 🏭 RECYCLE NET — Experimentation Pipeline

This notebook loads the pre-split CSVs produced by `01_dataset_preparation.ipynb`
and runs **10 experiments** (5 architectures × 2 variants: with and without GrabCut
segmentation) to compare classification performance across setups.

## Experiments
| # | Architecture | Segmentation |
|---|---|---|
| 1 | ResNet-18 | ❌ No |
| 2 | ResNet-18 | ✅ Yes |
| 3 | ResNet-50 | ❌ No |
| 4 | ResNet-50 | ✅ Yes |
| 5 | EfficientNet-B0 | ❌ No |
| 6 | EfficientNet-B0 | ✅ Yes |
| 7 | MobileNet-V3-Small | ❌ No |
| 8 | MobileNet-V3-Small | ✅ Yes |
| 9 | DenseNet-121 | ❌ No |
| 10 | DenseNet-121 | ✅ Yes |

Each experiment fine-tunes a frozen backbone with a new classification head,
then evaluates on validation and test splits. A summary DataFrame collects all results.

## 1. Libraries & Reproducibility

In [ ]:
import os
import random
import time

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import cv2
import kagglehub

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Seed fixed : {SEED}")
print(f"Device     : {DEVICE}")

## 2. Configuration

In [ ]:
# ── Execution mode ────────────────────────────────────────────────────────────
# "local"  : reads CSVs from ../data/ and images referenced inside them.
# "online" : downloads datasets via kagglehub (same as notebook 01).
EXECUTION_MODE = "local"   # change to "online" on Colab / Kaggle

# ── Training hyper-parameters ─────────────────────────────────────────────────
BATCH_SIZE  = 32
NUM_EPOCHS  = 10
LR          = 1e-3
NUM_CLASSES = 9

# ── Class mapping ─────────────────────────────────────────────────────────────
TARGET_CLASSES = {
    "glass": 0, "paper": 1, "cardboard": 2, "plastic": 3, "metal": 4,
    "trash": 5, "battery": 6, "biological": 7, "textile": 8,
}
IDX_TO_CLASS = {
    0: "Glass", 1: "Paper", 2: "Cardboard", 3: "Plastic", 4: "Metal",
    5: "Trash",  6: "Battery", 7: "Biological", 8: "Textile",
}

print(f"Execution mode : {EXECUTION_MODE}")
print(f"Batch size     : {BATCH_SIZE}")
print(f"Epochs         : {NUM_EPOCHS}")
print(f"Learning rate  : {LR}")
print(f"Num classes    : {NUM_CLASSES}")

## 3. Data Paths

**Local mode** – reads CSVs from `../data/` and resolves image paths stored in them.  
**Online mode** – downloads both Kaggle datasets with `kagglehub` and rebuilds the
train/val/test splits using the same stratified split as notebook 01.

In [ ]:
DATASETS_CONFIG_ONLINE = [
    {
        "path_key": "ds1",
        "mapping": {
            "glass": "glass", "paper": "paper", "cardboard": "cardboard",
            "plastic": "plastic", "metal": "metal", "trash": None,
        },
    },
    {
        "path_key": "ds2",
        "mapping": {
            "brown-glass": "glass", "green-glass": "glass", "white-glass": "glass",
            "paper": "paper", "cardboard": "cardboard", "plastic": "plastic",
            "metal": "metal", "battery": "battery", "biological": "biological",
            "clothes": "textile", "shoes": "textile",
            "unknown": None, "food-waste": None,
        },
    },
]


def build_samples_from_roots(path_ds1: str, path_ds2: str) -> list:
    paths_map = {"ds1": path_ds1, "ds2": path_ds2}
    samples = []
    for cfg in DATASETS_CONFIG_ONLINE:
        root = paths_map[cfg["path_key"]]
        for dirpath, _, files in os.walk(root):
            cls_name = os.path.basename(dirpath).lower()
            if cls_name not in cfg["mapping"]:
                continue
            target = cfg["mapping"][cls_name]
            if target is None:
                continue
            label = TARGET_CLASSES[target]
            for f in files:
                if f.lower().endswith((".jpg", ".jpeg", ".png")):
                    samples.append((os.path.join(dirpath, f), label))
    return samples


if EXECUTION_MODE == "local":
    print("📦 Loading local CSVs …")
    train_df = pd.read_csv("../data/train.csv")
    val_df   = pd.read_csv("../data/val.csv")
    test_df  = pd.read_csv("../data/test.csv")

    # Paths in the CSV may be relative to ../data/raw/
    RAW_DIR = os.path.abspath("../data/raw")
    for df in (train_df, val_df, test_df):
        df["path"] = df["path"].apply(
            lambda p: p if os.path.isabs(p) else os.path.join(RAW_DIR, p)
        )

else:
    print("📦 Downloading Dataset 1 (Base) …")
    path_ds1 = kagglehub.dataset_download("zlatan599/garbage-dataset-classification")
    print("📦 Downloading Dataset 2 (Industrial) …")
    path_ds2 = kagglehub.dataset_download("mostafaabla/garbage-classification")

    from sklearn.model_selection import train_test_split as _tts

    all_samples = build_samples_from_roots(path_ds1, path_ds2)
    all_paths   = [s[0] for s in all_samples]
    all_labels  = [s[1] for s in all_samples]
    idx_all     = list(range(len(all_samples)))

    train_idx, temp_idx = _tts(idx_all, test_size=0.2, stratify=all_labels, random_state=SEED)
    temp_labels = [all_labels[i] for i in temp_idx]
    val_idx, test_idx = _tts(temp_idx, test_size=0.5, stratify=temp_labels, random_state=SEED)

    train_df = pd.DataFrame({"path": [all_paths[i] for i in train_idx], "label": [all_labels[i] for i in train_idx]})
    val_df   = pd.DataFrame({"path": [all_paths[i] for i in val_idx],   "label": [all_labels[i] for i in val_idx]})
    test_df  = pd.DataFrame({"path": [all_paths[i] for i in test_idx],  "label": [all_labels[i] for i in test_idx]})

print(f"\n✅ Data loaded.")
print(f"   Train : {len(train_df):>5} samples")
print(f"   Val   : {len(val_df):>5} samples")
print(f"   Test  : {len(test_df):>5} samples")

## 4. GrabCut Segmentation Helper

Same GrabCut logic used in `01_dataset_preparation.ipynb`.  
Returns a PIL Image with the background zeroed out.

In [ ]:
def grabcut_segment(image: Image.Image) -> Image.Image:
    """Apply GrabCut segmentation; background pixels are set to black."""
    image   = image.resize((224, 224))
    img_np  = np.array(image)

    mask      = np.zeros(img_np.shape[:2], np.uint8)
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)

    h, w = img_np.shape[:2]
    rect = (10, 10, w - 20, h - 20)

    cv2.grabCut(img_np, mask, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_RECT)

    binary    = np.where((mask == 2) | (mask == 0), 0, 1).astype("uint8")
    segmented = (img_np * binary[:, :, np.newaxis]).astype(np.uint8)
    return Image.fromarray(segmented)


# ── Quick visual demo ─────────────────────────────────────────────────────────
demo_rows = train_df.sample(3, random_state=SEED).reset_index(drop=True)

fig, axes = plt.subplots(3, 2, figsize=(8, 10))
fig.suptitle("GrabCut Segmentation Demo", fontsize=13, fontweight="bold")

for i, row in demo_rows.iterrows():
    orig = Image.open(row["path"]).convert("RGB")
    seg  = grabcut_segment(orig)

    axes[i, 0].imshow(orig.resize((224, 224)))
    axes[i, 0].set_title(f"Original — {IDX_TO_CLASS[row['label']]}", fontsize=9)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(seg)
    axes[i, 1].set_title("Segmented", fontsize=9)
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

## 5. Dataset & DataLoader Factory

`CSVGarbageDataset` reads images from a split DataFrame.  
The `use_segmentation` flag applies GrabCut before the torchvision transforms.

In [ ]:
class CSVGarbageDataset(Dataset):
    """
    Dataset backed by a split DataFrame with columns [path, label].

    Args:
        df               : DataFrame with image paths and integer labels.
        transform        : torchvision Compose pipeline.
        use_segmentation : if True, GrabCut is applied before the transform.
    """

    def __init__(self, df: pd.DataFrame, transform=None, use_segmentation: bool = False):
        self.df               = df.reset_index(drop=True)
        self.transform        = transform
        self.use_segmentation = use_segmentation

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row   = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")

        if self.use_segmentation:
            image = grabcut_segment(image)

        if self.transform:
            image = self.transform(image)

        return image, int(row["label"])


# ── Transforms ────────────────────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def make_loaders(use_segmentation: bool):
    """Return (train_loader, val_loader, test_loader)."""
    tr = CSVGarbageDataset(train_df, train_transforms,    use_segmentation)
    va = CSVGarbageDataset(val_df,   val_test_transforms, use_segmentation)
    te = CSVGarbageDataset(test_df,  val_test_transforms, use_segmentation)

    kw_tr = dict(batch_size=BATCH_SIZE, shuffle=True,  drop_last=True,  num_workers=0, pin_memory=True)
    kw_ev = dict(batch_size=BATCH_SIZE, shuffle=False, num_workers=0,   pin_memory=True)

    return DataLoader(tr, **kw_tr), DataLoader(va, **kw_ev), DataLoader(te, **kw_ev)


print("✅ Dataset and DataLoader factory ready.")

## 6. Model Factory

Five ImageNet-pre-trained architectures with the classification head replaced.
All backbone parameters are **frozen** — only the new head is trainable.

In [ ]:
def build_model(arch: str, num_classes: int = NUM_CLASSES) -> nn.Module:
    """
    Build a frozen-backbone model with a new classification head.

    Supported: resnet18, resnet50, efficientnet_b0,
               mobilenet_v3_small, densenet121.
    """
    weights_map = {
        "resnet18":           models.ResNet18_Weights.IMAGENET1K_V1,
        "resnet50":           models.ResNet50_Weights.IMAGENET1K_V2,
        "efficientnet_b0":    models.EfficientNet_B0_Weights.IMAGENET1K_V1,
        "mobilenet_v3_small": models.MobileNet_V3_Small_Weights.IMAGENET1K_V1,
        "densenet121":        models.DenseNet121_Weights.IMAGENET1K_V1,
    }
    constructors = {
        "resnet18":           models.resnet18,
        "resnet50":           models.resnet50,
        "efficientnet_b0":    models.efficientnet_b0,
        "mobilenet_v3_small": models.mobilenet_v3_small,
        "densenet121":        models.densenet121,
    }

    if arch not in constructors:
        raise ValueError(f"Unknown architecture: {arch}")

    model = constructors[arch](weights=weights_map[arch])

    # Freeze backbone
    for param in model.parameters():
        param.requires_grad = False

    # Replace head
    if arch in ("resnet18", "resnet50"):
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif arch == "efficientnet_b0":
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    elif arch == "mobilenet_v3_small":
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)

    elif arch == "densenet121":
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)

    return model.to(DEVICE)


# Parameter overview
print(f"{'Architecture':<22} {'Total params':>14} {'Trainable':>12}")
print("-" * 50)
for arch in ["resnet18", "resnet50", "efficientnet_b0", "mobilenet_v3_small", "densenet121"]:
    m = build_model(arch)
    total     = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{arch:<22} {total:>14,} {trainable:>12,}")

## 7. Training & Evaluation Utilities

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    """One training epoch. Returns (avg_loss, accuracy)."""
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted  = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Evaluation pass. Returns (avg_loss, accuracy)."""
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss    = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, predicted  = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def get_all_preds_labels(model, loader):
    """Collect predictions and true labels over a full DataLoader."""
    model.eval()
    all_preds, all_labels = [], []

    for images, labels in loader:
        images = images.to(DEVICE)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

    return np.array(all_preds), np.array(all_labels)


print("✅ Training utilities ready.")

## 8. Experiment Runner

`run_experiment()` trains one (architecture, segmentation) pair and returns
a results dict with training history and test-set metrics.

In [ ]:
def run_experiment(arch: str, use_segmentation: bool, num_epochs: int = NUM_EPOCHS) -> dict:
    """
    Train and evaluate one experiment variant.

    Returns a dict with: name, arch, segmentation flag, history,
    best_val_acc, test_loss, test_acc, test_preds, test_labels, elapsed_sec.
    """
    seg_label = "segmented" if use_segmentation else "no_seg"
    exp_name  = f"{arch}__{seg_label}"

    print(f"\n{'='*62}")
    print(f"  Experiment : {exp_name}")
    print(f"{'='*62}")

    train_loader, val_loader, test_loader = make_loaders(use_segmentation)

    model     = build_model(arch)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=LR
    )
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    history      = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0
    best_weights = None
    t0 = time.time()

    for epoch in range(1, num_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_acc = evaluate(model, val_loader, criterion)
        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)

        print(
            f"  Epoch {epoch:>2}/{num_epochs} | "
            f"train loss: {tr_loss:.4f}  acc: {tr_acc:.4f} | "
            f"val loss: {va_loss:.4f}  acc: {va_acc:.4f}"
        )

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    elapsed = time.time() - t0

    # Restore best checkpoint and evaluate on test
    model.load_state_dict(best_weights)
    te_loss, te_acc = evaluate(model, test_loader, criterion)
    preds, labels   = get_all_preds_labels(model, test_loader)

    print(f"\n  ✅ Best val acc : {best_val_acc:.4f}")
    print(f"  ✅ Test acc     : {te_acc:.4f}")
    print(f"  ⏱  Time elapsed : {elapsed:.0f}s")

    return {
        "name":         exp_name,
        "arch":         arch,
        "segmentation": use_segmentation,
        "history":      history,
        "best_val_acc": best_val_acc,
        "test_loss":    te_loss,
        "test_acc":     te_acc,
        "test_preds":   preds,
        "test_labels":  labels,
        "elapsed_sec":  elapsed,
    }


print("✅ Experiment runner defined.")

## 9. Run All 10 Experiments

> ⚠️ **Runtime note**: on CPU this takes several hours. Use a GPU environment or
> reduce `NUM_EPOCHS` in Section 2 for a quick smoke-test.

In [ ]:
ARCHITECTURES = [
    "resnet18",
    "resnet50",
    "efficientnet_b0",
    "mobilenet_v3_small",
    "densenet121",
]

results = []

for arch in ARCHITECTURES:
    for use_seg in [False, True]:   # False first (baseline), True second (segmented)
        res = run_experiment(arch, use_segmentation=use_seg)
        results.append(res)

print("\n🎉 All 10 experiments finished.")

## 10. Results Summary Table

In [ ]:
summary_df = pd.DataFrame([
    {
        "Experiment":     r["name"],
        "Architecture":   r["arch"],
        "Segmentation":   "Yes" if r["segmentation"] else "No",
        "Best Val Acc %": round(r["best_val_acc"] * 100, 2),
        "Test Acc %":     round(r["test_acc"]     * 100, 2),
        "Test Loss":      round(r["test_loss"],          4),
        "Train Time (s)": int(r["elapsed_sec"]),
    }
    for r in results
]).sort_values("Test Acc %", ascending=False).reset_index(drop=True)

print("=" * 72)
print("EXPERIMENT SUMMARY  (sorted by Test Accuracy ↓)")
print("=" * 72)
print(summary_df.to_string(index=False))

best = summary_df.iloc[0]
print(f"\n🏆 Best overall : {best['Experiment']}  →  Test Acc = {best['Test Acc %']:.2f}%")

## 11. Learning Curves per Architecture

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(16, 22))
fig.suptitle("Learning Curves — Val Accuracy per Experiment", fontsize=14, fontweight="bold")

for row_idx, arch in enumerate(ARCHITECTURES):
    arch_results = [r for r in results if r["arch"] == arch]
    for col_idx, res in enumerate(arch_results):
        ax     = axes[row_idx, col_idx]
        epochs = range(1, len(res["history"]["val_acc"]) + 1)

        ax.plot(epochs, res["history"]["train_acc"], label="Train Acc", linewidth=1.8)
        ax.plot(epochs, res["history"]["val_acc"],   label="Val Acc",   linewidth=1.8, linestyle="--")
        ax.set_title(res["name"].replace("__", "  |  "), fontsize=9, fontweight="bold")
        ax.set_xlabel("Epoch", fontsize=8)
        ax.set_ylabel("Accuracy", fontsize=8)
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
        ax.set_xlim(1, NUM_EPOCHS)

plt.tight_layout()
plt.show()

## 12. Test Accuracy: Segmentation vs No Segmentation

In [ ]:
x      = np.arange(len(ARCHITECTURES))
w      = 0.35
no_seg  = [r["test_acc"] * 100 for r in results if not r["segmentation"]]
yes_seg = [r["test_acc"] * 100 for r in results if     r["segmentation"]]

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - w/2, no_seg,  w, label="No Segmentation",   color="#4C72B0", edgecolor="white")
bars2 = ax.bar(x + w/2, yes_seg, w, label="With Segmentation", color="#55A868", edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(ARCHITECTURES, rotation=15, ha="right", fontsize=10)
ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Test Accuracy: Segmentation vs No Segmentation", fontsize=12, fontweight="bold")
ax.legend()
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

# Delta table
print("\nImpact of segmentation on test accuracy:")
print(f"{'Architecture':<22} {'No Seg':>8} {'Seg':>8} {'Delta':>8}")
print("-" * 50)
for arch, ns, ys in zip(ARCHITECTURES, no_seg, yes_seg):
    delta = ys - ns
    sign  = "+" if delta >= 0 else ""
    print(f"{arch:<22} {ns:>8.2f}% {ys:>8.2f}%  {sign}{delta:.2f}%")

## 13. Confusion Matrix — Best Experiment

In [ ]:
best_result  = max(results, key=lambda r: r["test_acc"])
preds_best   = best_result["test_preds"]
labels_best  = best_result["test_labels"]
class_names_list = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]

cm = confusion_matrix(labels_best, preds_best)
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names_list).plot(
    ax=ax, cmap="Blues", colorbar=True, xticks_rotation=45
)
ax.set_title(
    f"Confusion Matrix — {best_result['name']}  (Test Acc: {best_result['test_acc']*100:.2f}%)",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.show()

## 14. Classification Report — Best Experiment

In [ ]:
print(f"Classification Report — {best_result['name']}\n")
print(classification_report(labels_best, preds_best, target_names=class_names_list, digits=4))

## 15. Confusion Matrices — All Experiments

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(28, 11))
fig.suptitle("Confusion Matrices — All 10 Experiments", fontsize=14, fontweight="bold")

for ax, res in zip(axes.flatten(), results):
    cm_i = confusion_matrix(res["test_labels"], res["test_preds"])
    ConfusionMatrixDisplay(confusion_matrix=cm_i, display_labels=class_names_list).plot(
        ax=ax, cmap="Blues", colorbar=False, xticks_rotation=90
    )
    ax.set_title(f"{res['name']}\nAcc: {res['test_acc']*100:.1f}%", fontsize=8, fontweight="bold")
    ax.tick_params(labelsize=6)

plt.tight_layout()
plt.show()

## 16. Save Results to CSV

In [ ]:
os.makedirs("../data", exist_ok=True)
summary_df.to_csv("../data/experiment_results.csv", index=False)
print("✅ Results saved to ../data/experiment_results.csv")
print()
print(summary_df.to_string(index=False))